# Hafta 1 — Veri İşleme + Model Altyapısı
**Sultan** — AteşKes (EERİS+)

Bu notebook Hafta 1 teslimini içerir: OSM'den çekilen kritik alanlar (hastane/okul/huzurevi/köy) için yükseklik/eğim verisi, 4 faktörlü `risk_skoru` ve `oncelik_skoru` hesaplanmış bir DataFrame.

In [1]:
import time
import requests
import pandas as pd
from hesaplamalar import egim_hesapla, egim_yonu_belirle, risk_skoru, oncelik_skoru

## 1. OSM'den kritik alanları çekme (Overpass API)
Muğla için hastane, okul, huzurevi, köy noktalarını çekiyoruz. Overpass'ın genel sunucusu zaman zaman 504 (gateway timeout) döndürebiliyor; bu yüzden birkaç deneme yapan basit bir retry ekledik.

In [2]:
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
HEADERS = {"User-Agent": "AtesKesProject/1.0 (Biz Teknopark on kuluçka)"}
TIP_ESLEME = {"hospital": "hastane", "school": "okul", "nursing_home": "huzurevi", "village": "koy"}


def osm_kritik_alanlari_cek(il_adi="Muğla", limit=6):
    sorgu = f"""
    [out:json][timeout:60];
    area["name"="{il_adi}"]["admin_level"="4"]->.aranan;
    (
      node["amenity"="hospital"](area.aranan);
      node["amenity"="school"](area.aranan);
      node["amenity"="nursing_home"](area.aranan);
      node["place"="village"](area.aranan);
    );
    out center;
    """
    son_hata = None
    for deneme in range(4):
        try:
            r = requests.post(OVERPASS_URL, data={"data": sorgu}, headers=HEADERS, timeout=90)
            r.raise_for_status()
            break
        except requests.exceptions.HTTPError as e:
            son_hata = e
            print(f"  Overpass denemesi {deneme+1} başarısız ({e}), tekrar deneniyor...")
            time.sleep(5)
    else:
        raise son_hata
    elemanlar = r.json()["elements"]

    tip_gruplari = {}
    for e in elemanlar:
        tip = e["tags"].get("amenity") or e["tags"].get("place")
        tip_tr = TIP_ESLEME.get(tip, tip)
        tip_gruplari.setdefault(tip_tr, []).append({
            "isim": e["tags"].get("name", "isimsiz"),
            "tip": tip_tr,
            "lat": e["lat"],
            "lon": e["lon"],
        })

    # Çeşitlilik için her tipten sırayla al (hepsi aynı tip olmasın)
    sonuc = []
    tur_listeleri = list(tip_gruplari.values())
    i = 0
    while len(sonuc) < limit and any(tur_listeleri):
        tur = tur_listeleri[i % len(tur_listeleri)]
        if tur:
            sonuc.append(tur.pop(0))
        i += 1
        if i > limit * 10:
            break
    return sonuc[:limit]


kritik_alanlar_ham = osm_kritik_alanlari_cek(limit=6)
print(f"{len(kritik_alanlar_ham)} kritik alan bulundu.")
kritik_alanlar_ham

6 kritik alan bulundu.


[{'isim': 'İslamhaneleri', 'tip': 'koy', 'lat': 37.029895, 'lon': 27.2947027},
 {'isim': 'Bodrum Amerikan Hastanesi',
  'tip': 'hastane',
  'lat': 37.0399394,
  'lon': 27.4289622},
 {'isim': 'Kemer İlköğretim Okulu',
  'tip': 'okul',
  'lat': 36.64692,
  'lon': 29.3620705},
 {'isim': 'İkinci Bahar Huzur Evi',
  'tip': 'huzurevi',
  'lat': 36.8552761,
  'lon': 28.279101},
 {'isim': 'Gürece', 'tip': 'koy', 'lat': 37.0395376, 'lon': 27.3224322},
 {'isim': 'isimsiz', 'tip': 'hastane', 'lat': 36.6223089, 'lon': 29.1150102}]

## 2. Yükseklik verisi (Open-Topo-Data) ve eğim hesabı
Her nokta için merkez + 4 komşu (~100m kuzey/güney/doğu/batı) yükseklik sorgulanıyor, `egim_hesapla()` ile eğim dereceye çevriliyor.

In [3]:
TOPO_URL = "https://api.opentopodata.org/v1/srtm30m"


def yukseklik_verisi_cek(lat, lon, offset=0.0009):
    noktalar = {
        "merkez": (lat, lon),
        "kuzey": (lat + offset, lon),
        "guney": (lat - offset, lon),
        "dogu": (lat, lon + offset),
        "bati": (lat, lon - offset),
    }
    locations = "|".join(f"{p[0]},{p[1]}" for p in noktalar.values())
    r = requests.get(TOPO_URL, params={"locations": locations}, timeout=30)
    r.raise_for_status()
    sonuclar = r.json()["results"]
    return {ad: s["elevation"] for ad, s in zip(noktalar.keys(), sonuclar)}

## 3. Hava durumu (Open-Meteo)
Risk formülü sıcaklık/rüzgar/nem gerektiriyor; Esma'nın backend'i hazır olana kadar aynı ücretsiz Open-Meteo API'sini burada da kullanıyoruz.

In [4]:
METEO_URL = "https://api.open-meteo.com/v1/forecast"


def hava_durumu_cek(lat, lon):
    params = {"latitude": lat, "longitude": lon, "current": "temperature_2m,relative_humidity_2m,wind_speed_10m"}
    r = requests.get(METEO_URL, params=params, timeout=30)
    r.raise_for_status()
    c = r.json()["current"]
    return c["temperature_2m"], c["wind_speed_10m"], c["relative_humidity_2m"]


sicaklik, ruzgar_hizi, nem = hava_durumu_cek(37.2153, 28.3636)  # Muğla merkez
print(f"sicaklik={sicaklik} ruzgar_hizi={ruzgar_hizi} nem={nem}")

sicaklik=27.4 ruzgar_hizi=19.5 nem=16


## 4. Hafta 1 teslimi: risk_skoru + oncelik_skoru DataFrame'i
Ortak veri sözleşmesindeki alan isimlerine uygun şekilde birleştiriyoruz.

In [5]:
kayitlar = []
for i, alan in enumerate(kritik_alanlar_ham):
    print(f"[{i+1}/{len(kritik_alanlar_ham)}] yukseklik cekiliyor: {alan['isim']}")
    yukseklikler = yukseklik_verisi_cek(alan["lat"], alan["lon"])
    egim = egim_hesapla(yukseklikler)
    egim_yonu = egim_yonu_belirle(yukseklikler)  # yokus yukari yon (kuzey/guney/dogu/bati)
    risk = risk_skoru(sicaklik, ruzgar_hizi, nem, egim)
    oncelik = oncelik_skoru(alan["tip"])
    kayitlar.append({
        "bolge_id": f"mugla_{i+1:02d}",
        "isim": alan["isim"],
        "tip": alan["tip"],
        "lat": alan["lat"],
        "lon": alan["lon"],
        "yukseklik_metre": yukseklikler["merkez"],
        "egim_derece": egim,
        "egim_yonu": egim_yonu,
        "risk_skoru": round(risk, 3),
        "oncelik_skoru": oncelik,
    })
    time.sleep(1)  # Open-Topo-Data ucretsiz katman limiti

kritik_alanlar_df = pd.DataFrame(kayitlar)
kritik_alanlar_df

[1/6] yukseklik cekiliyor: İslamhaneleri


[2/6] yukseklik cekiliyor: Bodrum Amerikan Hastanesi


[3/6] yukseklik cekiliyor: Kemer İlköğretim Okulu


[4/6] yukseklik cekiliyor: İkinci Bahar Huzur Evi


[5/6] yukseklik cekiliyor: Gürece


[6/6] yukseklik cekiliyor: isimsiz


,bolge_id,isim,tip,lat,lon,yukseklik_metre,egim_derece,egim_yonu,risk_skoru,oncelik_skoru
0,mugla_01,İslamhaneleri,koy,37.029895,27.294703,40.0,11.9,kuzey,0.135,0.7
1,mugla_02,Bodrum Amerikan Hastanesi,hastane,37.039939,27.428962,13.0,1.7,kuzey,0.099,1.0
2,mugla_03,Kemer İlköğretim Okulu,okul,36.646920,29.362071,124.0,1.1,kuzey,0.097,0.9
3,mugla_04,İkinci Bahar Huzur Evi,huzurevi,36.855276,28.279101,5.0,9.1,dogu,0.125,0.9
4,mugla_05,Gürece,koy,37.039538,27.322432,160.0,10.8,kuzey,0.131,0.7
5,mugla_06,isimsiz,hastane,36.622309,29.115010,5.0,2.3,guney,0.101,1.0


In [6]:
# Ortak veri sözleşmesindeki formata uyması ve Hafta 2'de kullanılması için kaydet
kritik_alanlar_df.to_csv("hafta1_ornek_veri.csv", index=False, encoding="utf-8")
print("Kaydedildi: hafta1_ornek_veri.csv")

Kaydedildi: hafta1_ornek_veri.csv


## Notlar / sonraki adım
- Bu ay eğim, 5 nokta bazlı kaba bir yaklaşımla hesaplanıyor (plan gereği bilinçli bir basitleştirme). 2. ayda gerçek DEM/SRTM rasteri (GDAL/richdem) ile değiştirilecek.
- `risk_skoru` ağırlıkları (0.25/0.25/0.15/0.35) şu an kural tabanlı; 2. ayda gerçek etiketli veriyle kalibre edilecek.
- Bu notebook'taki tüm fonksiyonlar `hesaplamalar.py` modülünde toplanmış durumda; Esma bunları FastAPI `/yangin-noktalari` ve `/bolgeler` endpoint'lerine bağlayabilir.